# Review BirdNET detections

Pull BirdNET detections, filter them to a few species/recordings, and review the clips in `jupyter_bioacoustic`.

How audio gets to the widget:

1. **Detections** come from a public JSON endpoint (`DETECTIONS_URL`). No auth needed.
2. **Recording metadata** comes from `GET /recordings/<id>` on the API (`API_URL`). It needs the session token cookie (`API_TOKEN` in `.env`). The response has an S3 `path` (`<bucket>/<key>`) and a `deployment_id`, which gives us the site.
3. **Audio**: `GET /media/<key>` (with the cookie) answers with a 307 redirect to a presigned S3 URL. `ondio.read_flac` reads only the bytes for each clip from that URL, so the multi-GB recordings are never downloaded.
4. **Widget**: the widget asks the kernel for audio by calling `jupyter_bioacoustic.audio.read_segment(path, ...)` with no auth. `install_widget_audio()` wraps that function so API recording URLs go through steps 2–3.

The helper functions live in [`api_audio.py`](api_audio.py). The session token expires. If the token check below fails, copy a fresh token into `.env` and rerun all cells.

In [21]:
import pandas as pd
import requests

import api_audio
from api_audio import query_api, read_clip

DETECTIONS_URL = "https://eshpezgjnn.us-west-2.awsapprunner.com/birdnet/latest/detections"
API_URL = "https://api.wildlifesoundhub.org"

api_audio.connect(API_URL, token_env="API_TOKEN", cookie_name="__Secure-authjs.session-token")

API token OK


## Detections

One row per (3 s window, predicted label), ranked by `rank` within each window. `start_time`/`end_time` are seconds from the start of the recording.

In [22]:
det = pd.DataFrame(requests.get(DETECTIONS_URL, timeout=120).json())
print(f"{len(det):,} detections across {det.recording_id.nunique()} recordings")
det.head()

134,870 detections across 4 recordings


,id,recording_id,common_name,scientific_name,label,confidence,rank,start_time,end_time
0,1_t0p0_r1,1,Texas Bush Katydid,Scudderia texensis,Scudderia texensis_Texas Bush Katydid,0.024407,1,0.0,3.0
1,1_t0p0_r2,1,Mallard,Anas platyrhynchos,Anas platyrhynchos_Mallard,0.023591,2,0.0,3.0
2,1_t0p0_r3,1,Southern Wood Cricket,Gryllus fultoni,Gryllus fultoni_Southern Wood Cricket,0.016493,3,0.0,3.0
3,1_t0p0_r4,1,Barn Owl,Tyto alba,Tyto alba_Barn Owl,0.012386,4,0.0,3.0
4,1_t3p0_r1,1,Tree Pipit,Anthus trivialis,Anthus trivialis_Tree Pipit,0.020336,1,3.0,6.0


## Recordings and sites

Detections have no site field, so we join recording → deployment (`placename` is the site).

In [23]:
recordings = pd.DataFrame(
    [query_api(f"/recordings/{rid}") for rid in sorted(det.recording_id.unique())]
).rename(columns={"id": "recording_id"})
deployments = pd.DataFrame(
    [query_api(f"/deployments/{did}") for did in recordings.deployment_id.unique()]
).rename(columns={"id": "deployment_id", "placename": "site", "code": "deployment_code"})

recordings = recordings.merge(
    deployments[["deployment_id", "site", "deployment_code"]], on="deployment_id", how="left"
)
recordings[["recording_id", "site", "deployment_code", "start", "end", "path"]]

,recording_id,site,deployment_code,start,end,path
0,1,AGCA1,AGCA1_2023,2023-05-22T00:00:00,2023-05-22T09:00:00,casoundhub/audio/cemaf-acoustics/AGCA1_2023/20...
1,2,AGCA1,AGCA1_2023,2023-05-22T20:00:00,2023-05-22T23:59:00,casoundhub/audio/cemaf-acoustics/AGCA1_2023/20...
2,3,AGCA1,AGCA1_2023,2023-05-23T00:00:00,2023-05-23T09:00:00,casoundhub/audio/cemaf-acoustics/AGCA1_2023/20...
3,4,AGCA1,AGCA1_2023,2023-05-23T20:00:00,2023-05-23T23:59:00,casoundhub/audio/cemaf-acoustics/AGCA1_2023/20...


In [24]:
# What's in there: top-ranked labels with confidence >= 0.5
(
    det.query("rank == 1 and confidence >= 0.5")
    .groupby(["common_name", "scientific_name"])
    .confidence.agg(["count", "max"])
    .sort_values("count", ascending=False)
    .head(25)
)

,,count,max
common_name,scientific_name,,
Western Meadowlark,Sturnella neglecta,154,0.869042
Lazuli Bunting,Passerina amoena,139,0.995642
Engine,Engine,101,0.967341
Ash-throated Flycatcher,Myiarchus cinerascens,59,0.932630
Southern Wood Cricket,Gryllus fultoni,40,0.785134
California Scrub-Jay,Aphelocoma californica,15,0.956025
Woodhouse's Scrub-Jay,Aphelocoma woodhouseii,12,0.840407
Tawny Owl,Strix aluco,10,0.927336
Rufous-crowned Sparrow,Aimophila ruficeps,8,0.827781


## Filter to the clips to review

In [25]:
SPECIES = ["Engine"]  # common names
SITES = None  # e.g. ["AGCA1"]; None = all
RECORDING_IDS = None  # e.g. [1, 3]; None = all
MIN_CONFIDENCE = 0.5
MAX_CLIPS = 50

clips = det.merge(recordings[["recording_id", "site"]], on="recording_id", how="left")
clips = clips[clips.common_name.isin(SPECIES) & (clips.confidence >= MIN_CONFIDENCE)]
if SITES is not None:
    clips = clips[clips.site.isin(SITES)]
if RECORDING_IDS is not None:
    clips = clips[clips.recording_id.isin(RECORDING_IDS)]
clips = clips.sort_values("confidence", ascending=False).head(MAX_CLIPS).reset_index(drop=True)

# The widget reads audio from this column; install_widget_audio() below resolves it to S3.
clips["audio_url"] = API_URL + "/recordings/" + clips.recording_id.astype(str)

print(f"{len(clips)} clips")
clips.head()

50 clips


,id,recording_id,common_name,scientific_name,label,confidence,rank,start_time,end_time,site,audio_url
0,3_t30489p0_r1,3,Engine,Engine,Engine_Engine,0.967341,1,30489.0,30492.0,AGCA1,https://api.wildlifesoundhub.org/recordings/3
1,4_t6174p0_r1,4,Engine,Engine,Engine_Engine,0.909348,1,6174.0,6177.0,AGCA1,https://api.wildlifesoundhub.org/recordings/4
2,1_t8067p0_r1,1,Engine,Engine,Engine_Engine,0.900637,1,8067.0,8070.0,AGCA1,https://api.wildlifesoundhub.org/recordings/1
3,4_t12924p0_r1,4,Engine,Engine,Engine_Engine,0.888116,1,12924.0,12927.0,AGCA1,https://api.wildlifesoundhub.org/recordings/4
4,4_t12954p0_r1,4,Engine,Engine,Engine_Engine,0.866854,1,12954.0,12957.0,AGCA1,https://api.wildlifesoundhub.org/recordings/4


## Audio access: API → presigned S3 → ondio ranged reads

In [26]:
# Check: read the top clip directly and play it
from IPython.display import Audio

row = clips.iloc[0]
samples, sr = read_clip(row.recording_id, row.start_time, row.end_time)
print(row.common_name, row.confidence, samples.shape, sr)
Audio(samples[:, 0], rate=sr)

Engine 0.967341423034668 (144000, 1) 48000


In [27]:
api_audio.install_widget_audio()

## Review

In [28]:
from jupyter_bioacoustic import BioacousticAnnotator

ann = BioacousticAnnotator(
    data=clips,
    data_index_column="id",
    audio_column="audio_url",
    display_columns=["common_name", "confidence", "rank", "site", "recording_id", "start_time", "end_time"],
    info_card_title="[[common_name]]",
    default_buffer=2,  # seconds of padding on each side of the detection
    output="../data/reviews.csv",
)
print(ann.validate())
ann.open()

{'valid': True, 'errors': [], 'warnings': ['No form input elements configured']}


<IPython.core.display.Javascript object>